In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
TRAIN_FILE = "grade5_6_math_train.json"
VAL_FILE   = "grade5_6_math_val.json"

MAX_LEN = 512
BATCH_SIZE = 1
EPOCHS = 2
LR = 2e-5

NUM_SECTIONS = 8
DEVICE = "cuda"

In [ ]:
SECTION_INDEX = {
    "introduction": 0,
    "concept_explanation": 1,
    "worked_examples": 2,
    "practice_questions": 3,
    "word_problems": 4,
    "pacing": 5,
    "clarity": 6,
    "engagement": 7
}

In [ ]:
import json

def build_prompt(sample):
    if sample["task"] == "task_a":
        return f"""### TASK: ANSWER EVALUATION
Topic: {sample['topic']}
Question: {sample['question']}
Student Answer:
{sample['answer']}

Return:
"""
    else:
        time_lines = "\n".join(
            f"[{k.upper()}_TIME={v}]" for k, v in sample["time_spent"].items()
        )
        return f"""### TASK: TEACHING ANALYSIS
Teacher Guide:
{sample['teacher_guide']}

Student Feedback:
{sample['student_feedback']}

Time Signals:
{time_lines}

Return:
"""


def build_completion(sample):
    return json.dumps(sample["completion"], indent=2)

In [ ]:
import json
import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:01, ?B/s]

In [ ]:
def preprocess(sample):
    prompt = build_prompt(sample)
    completion = build_completion(sample)

    full_text = prompt + completion
    enc = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length"
    )

    prompt_len = len(tokenizer(prompt)["input_ids"])
    labels = enc["input_ids"].copy()
    labels[:prompt_len] = [-100] * prompt_len

    if sample["task"] == "task_a":
        score = torch.tensor(sample["score"], dtype=torch.bfloat16)
    else:
        score = torch.tensor(-1.0, dtype=torch.bfloat16)

    section_labels = torch.zeros(NUM_SECTIONS, dtype=torch.bfloat16)
    if sample["task"] == "task_b" and sample.get("weak_sections"):
        for s in sample["weak_sections"]:
            section_labels[SECTION_INDEX[s]] = 1.0

    return {
        "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
        "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "score": score,
        "section_labels": section_labels
    }

In [ ]:
from datasets import load_dataset

train_ds = load_dataset("json", data_files=TRAIN_FILE)["train"].map(preprocess)
val_ds   = load_dataset("json", data_files=VAL_FILE)["train"].map(preprocess)

train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels', 'score', 'section_labels'])
val_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels', 'score', 'section_labels'])

In [ ]:
import torch.nn as nn
from transformers import AutoModelForCausalLM

class MultiTaskModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model
        hidden = base_model.config.hidden_size

        self.score_head = nn.Sequential(
            nn.Linear(hidden, 1),
            nn.Sigmoid()
        )
        self.section_head = nn.Linear(hidden, NUM_SECTIONS)

    def forward(self, input_ids, attention_mask, labels=None,
                score=None, section_labels=None):

        out = self.base(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            output_hidden_states=True
        )

        # last real token
        seq_len = attention_mask.sum(dim=1) - 1
        hidden = out.hidden_states[-1][
            torch.arange(input_ids.size(0)), seq_len
        ]
        # Ensure hidden state is bfloat16 to match head dtypes
        hidden = hidden.to(torch.bfloat16)

        # Ensure base model's loss is also bfloat16 for consistent arithmetic
        loss = None
        if out.loss is not None:
            loss = out.loss.to(torch.bfloat16)

        score_pred = self.score_head(hidden).squeeze(-1)
        section_logits = self.section_head(hidden)

        if score is not None and score.min() >= 0:
            loss += nn.functional.mse_loss(score_pred, score)

        if section_labels is not None:
            loss += nn.functional.binary_cross_entropy_with_logits(
                section_logits, section_labels
            )

        return {
            "loss": loss,
            "score": score_pred,
            "sections": section_logits
        }

In [ ]:
! pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 16.6 MB/s eta 0:00:00


In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map="auto"
)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM"
)

base = get_peft_model(base, lora)
model = MultiTaskModel(base).to(DEVICE)
# Convert the heads to bfloat16 to match the base model's output dtype
model.score_head = model.score_head.to(torch.bfloat16)
model.section_head = model.section_head.to(torch.bfloat16)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
from torch.utils.data import DataLoader

def collate(batch):
    return {
        k: torch.stack([b[k] for b in batch]).to(DEVICE)
        for k in batch[0]
    }

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader   = DataLoader(val_ds, batch_size=1, collate_fn=collate)


In [ ]:
from torch.optim import AdamW
from torch.cuda.amp import autocast

optimizer = AdamW(model.parameters(), lr=LR)

model.train()
for epoch in range(EPOCHS):
    for step, batch in enumerate(train_loader):
        with autocast(dtype=torch.bfloat16):
            out = model(**batch)
            loss = out["loss"]

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if step % 50 == 0:
            print(f"Epoch {epoch} Step {step} Loss {loss.item():.4f}")

/tmp/ipython-input-1744586497.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16):


Epoch 0 Step 0 Loss 22.3750
Epoch 0 Step 50 Loss 17.1250
Epoch 0 Step 100 Loss 0.9062
Epoch 0 Step 150 Loss 0.3457
Epoch 0 Step 200 Loss 0.2754
Epoch 0 Step 250 Loss 0.2305
Epoch 0 Step 300 Loss 0.5586
Epoch 0 Step 350 Loss 0.6328
Epoch 0 Step 400 Loss 0.1836
Epoch 0 Step 450 Loss 0.1826
Epoch 0 Step 500 Loss 0.1895
Epoch 0 Step 550 Loss 0.1582
Epoch 0 Step 600 Loss 0.0933
Epoch 0 Step 650 Loss 0.0889
Epoch 0 Step 700 Loss 0.5430
Epoch 0 Step 750 Loss 0.0723
Epoch 0 Step 800 Loss 0.0559
Epoch 0 Step 850 Loss 0.0947
Epoch 0 Step 900 Loss 0.4961
Epoch 0 Step 950 Loss 0.0708
Epoch 0 Step 1000 Loss 0.5820
Epoch 0 Step 1050 Loss 0.0732


/tmp/ipython-input-1744586497.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16):


Epoch 1 Step 0 Loss 0.1089


KeyboardInterrupt: 

In [ ]:
# from google.colab import drive
# drive.mount('/content/gdrive')
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')


# Define the path on Google Drive where you want to save the model
DRIVE_MODEL_PATH = "/content/gdrive/MyDrive/teachers_guide_and_question_answer_model"

# Save the model and tokenizer to Google Drive
model.base.save_pretrained(DRIVE_MODEL_PATH)
tokenizer.save_pretrained(DRIVE_MODEL_PATH)

Mounted at /content/gdrive


('/content/gdrive/MyDrive/teachers_guide_and_question_answer_model/tokenizer_config.json',
 '/content/gdrive/MyDrive/teachers_guide_and_question_answer_model/special_tokens_map.json',
 '/content/gdrive/MyDrive/teachers_guide_and_question_answer_model/chat_template.jinja',
 '/content/gdrive/MyDrive/teachers_guide_and_question_answer_model/vocab.json',
 '/content/gdrive/MyDrive/teachers_guide_and_question_answer_model/merges.txt',
 '/content/gdrive/MyDrive/teachers_guide_and_question_answer_model/added_tokens.json',
 '/content/gdrive/MyDrive/teachers_guide_and_question_answer_model/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch
import os

# Ensure Google Drive is mounted
# from google.colab import drive
# drive.mount('/content/gdrive')

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount('/content/drive')

# Define the path on Google Drive where you saved the model
DRIVE_MODEL_PATH = "/content/gdrive/MyDrive/teachers_guide_and_question_answer_model"

# 1. Load the tokenizer
loaded_tokenizer = AutoTokenizer.from_pretrained(DRIVE_MODEL_PATH)

loaded_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)


loaded_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=loaded_bnb_config,
    device_map="auto"
)

loaded_peft_model = PeftModel.from_pretrained(loaded_base_model, DRIVE_MODEL_PATH)

loaded_multi_task_model = MultiTaskModel(loaded_peft_model).to(DEVICE)
# Convert the loaded model's custom heads to bfloat16 to match the base model's output dtype
loaded_multi_task_model.score_head = loaded_multi_task_model.score_head.to(torch.bfloat16)
loaded_multi_task_model.section_head = loaded_multi_task_model.section_head.to(torch.bfloat16)

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
@torch.no_grad()
def infer(sample):
    prompt = build_prompt(sample)
    ids = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    gen = loaded_multi_task_model.base.generate(
        **ids,
        max_new_tokens=200,
        temperature=0.8,
        top_p=0.95
    )

    outputs = loaded_multi_task_model(
    input_ids=ids["input_ids"],
    attention_mask=ids["attention_mask"]
)

    return {
        "generated_output": tokenizer.decode(gen[0], skip_special_tokens=True),
        "matching_score": outputs["score"].item(),
        "section_probs": torch.sigmoid(outputs["sections"]).cpu().tolist()
    }

In [ ]:
sample_4 = {
    "task": "task_b",
    "teacher_guide": "Grade 5 lesson on fractions with clear explanations.",
    "student_feedback": "Everything was clear and easy to follow.",
    "time_spent": {
        "introduction": 60,
        "concept_explanation": 140,
        "worked_examples": 180,
        "practice_questions": 160,
        "word_problems": 160,
        "pacing": 70,
        "clarity": 60,
        "engagement": 90
    }
}


result = infer(sample_4)
display(result)

{'generated_output': '### TASK: TEACHING ANALYSIS\nTeacher Guide:\nGrade 5 lesson on fractions with clear explanations.\n\nStudent Feedback:\nEverything was clear and easy to follow.\n\nTime Signals:\n[INTRODUCTION_TIME=60]\n[CONCEPT_EXPLANATION_TIME=140]\n[WORKED_EXAMPLES_TIME=180]\n[PRACTICE_QUESTIONS_TIME=160]\n[WORD_PROBLEMS_TIME=160]\n[PACING_TIME=70]\n[CLARITY_TIME=60]\n[ENGAGEMENT_TIME=90]\n\nReturn:\n',
 'matching_score': 0.61328125,
 'section_probs': [[0.8828125,
   0.498046875,
   0.2119140625,
   0.6484375,
   0.9375,
   0.90234375,
   0.52734375,
   0.28125]]}

In [ ]:
result['section_probs'][0]


[0.8828125,
 0.498046875,
 0.2119140625,
 0.6484375,
 0.9375,
 0.90234375,
 0.52734375,
 0.28125]

In [ ]:
weak_sections = [
    name for name, idx in SECTION_INDEX.items()
    if result['section_probs'][0][idx] >= 0.7
]

In [ ]:
weak_sections

['introduction', 'word_problems', 'pacing']